Crear caminos enteros con las tripletas.  

En vez de tripletas sueltas formatear con todo el camino.

INICIALIZACION

In [1]:
import sys
from sentence_transformers import SentenceTransformer

c:\Users\andre\anaconda3\envs\kag_env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sys.path.append("../src")

In [18]:
from neo4j import GraphDatabase

In [4]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j
from graph_retrieval.funciones_graph_retrieval import (
    extraer_top_k_entities,
    formatear_tripletas,
)
from conexion_qdrant.conexion_qdrant import ConexionQdrant
from RAG_retrieval.funciones_RAG_retrieval import(
    extraer_info_nodes,
    extraer_info_points,
    filtrar_nodos_por_score,
    textos_para_prompt
)
from funciones_generales import build_prompt
from LLM_interaction import LLM_interaction_functions as llm_funcs
from metricas.metricas_2Wiki import (
    f1_score,
    exact_match_score,
    respuesta_en_nodos_encontrados,
    suporting_facts_en_subgrafo,
    metricas_totales
    )
from output_save.funciones_guardado import guardar_resultados, guardar_registro

# Load Data

In [170]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 100, "train")

In [173]:
ejemplo = dataset_2Wiki[6]
ejemplo

{'_id': 'd4f5fa380baf11ebab90acde48001122',
 'type': 'inference',
 'question': "Who is Rhescuporis I (Odrysian)'s paternal grandfather?",
 'context': '[["John Mackay (poet)", ["John Mackay( 1656\\u20131754), known as( The Blind Piper), was a Scottish Gaelic poet and composer, and the grandfather of William Ross."]], ["Kaya Alp", ["Kaya Alp was, according to Ottoman tradition, the son of K\\u0131z\\u0131l Bu\\u011fa and the father of Suleyman Shah, who was, in turn, the grandfather of Ertu\\u011frul, and the great grandfather of the Ottoman Empire founder, Osman I."]], ["Zhao Shoushan", ["Zhao Shoushan( 12 November 1894 \\u2013 20 June 1965) was a KMT general and later Chinese Communist Party politician.", "He is the grandfather of Zhao Leji."]], ["Abd al-Muttalib", ["Abd al- Muttalib Shaybah ibn Hashim( c. 497 \\u2013 578) was the grandfather of Islamic prophet Muhammad."]], ["John Westley", ["Rev. John Westley( 1636 \\u2013 78) was an English nonconformist minister.", "He was the gran

# Qdrant y Neo4j conexion

In [167]:
database_Neo = "2wiki.prueba1"
database_Neo4j = ConexionNeo4j(database_Neo)
qd_client = ConexionQdrant()

In [168]:
driver = GraphDatabase.driver(
            "bolt://localhost:7687",
            auth=("neo4j", "password"),
            database = "2wiki.prueba1"
        )

In [169]:
collection = "2wikimultihop_prueba1"
embed_model_st = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3852.30it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Extraccion Entidades

In [201]:
dataset_2Wiki[9]

{'_id': '586507fa0bd911eba7f7acde48001122',
 'type': 'compositional',
 'question': 'Where was the director of film The Fascist born?',
 'context': '[["Olav Aaraas", ["Olav Aaraas( born 10 July 1950) is a Norwegian historian and museum director.", "He was born in Fredrikstad.", "From 1982 to 1993 he was the director of Sogn Folk Museum, from 1993 to 2010 he was the director of Maihaugen and from 2001 he has been the director of the Norwegian Museum of Cultural History.", "In 2010 he was decorated with the Royal Norwegian Order of St. Olav."]], ["Brian Kennedy (gallery director)", ["Brian Patrick Kennedy( born 5 November 1961) is an Irish- born art museum director who has worked in Ireland and Australia, and now lives and works in the United States.", "He is currently the director of the Peabody Essex Museum.", "He was the director of the Toledo Museum of Art in Ohio from 2010 to 2019.", "He was the director of the Hood Museum of Art from 2005 to 2010, and the National Gallery of Austral

In [202]:
question = dataset_2Wiki[9]["question"]

In [178]:
vector_index_name = "entity_embedding_index"

In [203]:
entidades_encontradas = database_Neo4j.query_a_embedding(vector_index_name, embed_model_st, question, 5)
entidades_filtradas = extraer_top_k_entities(entidades_encontradas, 2)


# Extraccion Subgrafo

In [ ]:
nodos, relaciones = database_Neo4j.extraer_subgrafo(entidades_filtradas, 2)

In [14]:
relaciones

{'relaciones_entidad_The Falcon (film)': [{'destino': 'Vatroslav Mimica',
   'origen': 'The Falcon (film)',
   'relacion': 'director'},
  {'destino': 'Croatian',
   'origen': 'Vatroslav Mimica',
   'relacion': 'country_of_citizenship'},
  {'destino': 'Yugoslavia',
   'origen': 'Vatroslav Mimica',
   'relacion': 'country_of_citizenship'}],
 'relaciones_entidad_Valentin the Good': [{'destino': 'Martin Frič',
   'origen': 'Valentin the Good',
   'relacion': 'director'},
  {'destino': 'Czech',
   'origen': 'Martin Frič',
   'relacion': 'country_of_citizenship'}]}

FUNCION extraer_subgrafo desglosada

In [20]:
query_base = """
    MATCH (n:Entity)
    WHERE n.name IN $entidades
    CALL apoc.path.subgraphAll(n, {
        maxLevel: $k
    })
    YIELD nodes, relationships
    RETURN 
    [node IN nodes | {name: node.name}] AS nodes,
    [rel in relationships | {
    origen: startNode(rel).name,
    destino: endNode(rel).name,
    relacion: type(rel)
    }
    ] AS relationships
"""

n_saltos = 2

subgrafo_raw = driver.execute_query(query_base, entidades = entidades_filtradas, k = n_saltos)


In [21]:
subgrafo_raw

EagerResult(records=[<Record nodes=[{'name': 'The Falcon (film)'}, {'name': 'Vatroslav Mimica'}, {'name': 'Croatian'}, {'name': 'Yugoslavia'}] relationships=[{'destino': 'Vatroslav Mimica', 'origen': 'The Falcon (film)', 'relacion': 'director'}, {'destino': 'Croatian', 'origen': 'Vatroslav Mimica', 'relacion': 'country_of_citizenship'}, {'destino': 'Yugoslavia', 'origen': 'Vatroslav Mimica', 'relacion': 'country_of_citizenship'}]>, <Record nodes=[{'name': 'Valentin the Good'}, {'name': 'Martin Frič'}, {'name': 'Czech'}] relationships=[{'destino': 'Martin Frič', 'origen': 'Valentin the Good', 'relacion': 'director'}, {'destino': 'Czech', 'origen': 'Martin Frič', 'relacion': 'country_of_citizenship'}]>], summary=<neo4j._work.summary.ResultSummary object at 0x0000018A64972790>, keys=['nodes', 'relationships'])

In [28]:
subgrafo_raw[0][0]["relationships"]

[{'destino': 'Vatroslav Mimica',
  'origen': 'The Falcon (film)',
  'relacion': 'director'},
 {'destino': 'Croatian',
  'origen': 'Vatroslav Mimica',
  'relacion': 'country_of_citizenship'},
 {'destino': 'Yugoslavia',
  'origen': 'Vatroslav Mimica',
  'relacion': 'country_of_citizenship'}]

-   apoc.path.subgraphAll devuelve las tripletas por separado --> Habria que reconstruir despues.  
-   con consulta normal y devolviendo - path si se tienen los caminos completos (solo desde el nodo origen)

In [121]:
queryyyyy = """
        MATCH path = (n:Entity)-[*1..3]-(m)
        WHERE n.name IN ['Canada', 'America']
        RETURN
            elementId(n) AS ID,
            n.name AS entidad_inicial,
            [node IN nodes(path) | node.name] AS nodos_names,
            [rel IN relationships(path) | type(rel)] AS relaciones_types

"""

# queryyyyy = """
#         MATCH path = (n:Entity)-[*1..3]-(m)
#         WHERE n.name IN ['Canada']
#         RETURN path

# """

subgrafo_raw, summary, key = driver.execute_query(queryyyyy)

In [122]:
subgrafo_raw

[<Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:185' entidad_inicial='Canada' nodos_names=['Canada', 'Lesser Slave Lake'] relaciones_types=['country']>,
 <Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:185' entidad_inicial='Canada' nodos_names=['Canada', 'Mother Teresa High School'] relaciones_types=['country']>,
 <Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:204' entidad_inicial='America' nodos_names=['America', 'Frankie Laine'] relaciones_types=['country_of_citizenship']>,
 <Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:204' entidad_inicial='America' nodos_names=['America', 'Frankie Laine', 'Hummingbird'] relaciones_types=['country_of_citizenship', 'performer']>]

In [125]:
subgrafo_raw[0]['entidad_inicial']

'Canada'

Crear lista de salida con todas las relaciones

In [128]:
subgrafo_raw

[<Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:185' entidad_inicial='Canada' nodos_names=['Canada', 'Lesser Slave Lake'] relaciones_types=['country']>,
 <Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:185' entidad_inicial='Canada' nodos_names=['Canada', 'Mother Teresa High School'] relaciones_types=['country']>,
 <Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:204' entidad_inicial='America' nodos_names=['America', 'Frankie Laine'] relaciones_types=['country_of_citizenship']>,
 <Record ID='4:5f3ab2a6-fc16-41f0-8b93-47cc4573193c:204' entidad_inicial='America' nodos_names=['America', 'Frankie Laine', 'Hummingbird'] relaciones_types=['country_of_citizenship', 'performer']>]

In [130]:
lista_relaciones = []

In [131]:
for rec in subgrafo_raw:
    relacion = [rec["entidad_inicial"]]
    for rel, node in zip(rec["relaciones_types"], rec["nodos_names"][1:]):
        relacion.append(rel)
        relacion.append(node)
    lista_relaciones.append(relacion)

In [132]:
lista_relaciones

[['Canada', 'country', 'Lesser Slave Lake'],
 ['Canada', 'country', 'Mother Teresa High School'],
 ['America', 'country_of_citizenship', 'Frankie Laine'],
 ['America',
  'country_of_citizenship',
  'Frankie Laine',
  'performer',
  'Hummingbird']]

# NUEVA FUNCION RETRIEVAL SUBGRAFO

Para tener los caminos enteros de 2 saltos

In [180]:
# def extraer_subgrafo_completo(self, entidades, n_saltos):
def extraer_subgrafo_completo(driver, entidades, n_saltos):
    query_base = f"""
        MATCH path = (n:Entity)-[*1..{n_saltos}]-(m)
        WHERE n.name IN $entis
        RETURN
            elementId(n) AS ID,
            n.name AS entidad_inicial,
            [node IN nodes(path) | node.name] AS nodos_names,
            [rel IN relationships(path) | type(rel)] AS relaciones_types
        """

    subgrafo_raw, summary, key = driver.execute_query(query_base, entis = entidades)
    lista_relaciones = []
    for rec in subgrafo_raw:
        relacion = [rec["entidad_inicial"]]
        for rel, node in zip(rec["relaciones_types"], rec["nodos_names"][1:]):
            relacion.append(rel)
            relacion.append(node)
        lista_relaciones.append(relacion)

    return lista_relaciones

In [150]:
entidades = ['Canada', 'America']
n_saltos = 2

In [204]:
# subgrafo = extraer_subgrafo_completo(driver, entidades, n_saltos)
subgrafo = extraer_subgrafo_completo(driver, entidades_filtradas, n_saltos)

In [205]:
subgrafo

[['The Fascist', 'director', 'Luciano Salce'],
 ['The Fascist', 'director', 'Luciano Salce', 'place_of_birth', 'Rome'],
 ['The Da Vinci Code (film)', 'country_of_origin', 'American'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'Stephen Roberts (director)'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'Diane Gilliam Fisher'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'Alison Skipper'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'Ted Swinford'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'McAllister Hull'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'Oscar Apfel'],
 ['The Da Vinci Code (film)',
  'country_of_origin',
  'American',
  'country_of_citizenship',
  'Billie H

In [206]:
print("\n".join([(" -> ".join(rel)) for rel in subgrafo]))

The Fascist -> director -> Luciano Salce
The Fascist -> director -> Luciano Salce -> place_of_birth -> Rome
The Da Vinci Code (film) -> country_of_origin -> American
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Stephen Roberts (director)
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Diane Gilliam Fisher
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Alison Skipper
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Ted Swinford
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> McAllister Hull
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Oscar Apfel
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Billie Holiday
The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Lewis R. Foster
The

In [184]:
def formatear_tripletas_extendidas(relaciones):
    tripletas_formateadas = [(" -> ".join(rel)) for rel in relaciones]
            
    return tripletas_formateadas

In [207]:
tripletas_formateadas = formatear_tripletas_extendidas(subgrafo)

In [208]:
tripletas_formateadas

['The Fascist -> director -> Luciano Salce',
 'The Fascist -> director -> Luciano Salce -> place_of_birth -> Rome',
 'The Da Vinci Code (film) -> country_of_origin -> American',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Stephen Roberts (director)',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Diane Gilliam Fisher',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Alison Skipper',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Ted Swinford',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> McAllister Hull',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Oscar Apfel',
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Billie Holiday',
 'The Da Vinci Code (film) -> country_of_origin -> American -> cou

## Aplicar re-ranking a las tripletas extendidas

In [187]:
from sentence_transformers import CrossEncoder

In [188]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5116.74it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [209]:
pairs = [[question, relacion] for relacion in tripletas_formateadas ]

In [210]:
pairs

[['Where was the director of film The Fascist born?',
  'The Fascist -> director -> Luciano Salce'],
 ['Where was the director of film The Fascist born?',
  'The Fascist -> director -> Luciano Salce -> place_of_birth -> Rome'],
 ['Where was the director of film The Fascist born?',
  'The Da Vinci Code (film) -> country_of_origin -> American'],
 ['Where was the director of film The Fascist born?',
  'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Stephen Roberts (director)'],
 ['Where was the director of film The Fascist born?',
  'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Diane Gilliam Fisher'],
 ['Where was the director of film The Fascist born?',
  'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Alison Skipper'],
 ['Where was the director of film The Fascist born?',
  'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Ted Sw

In [211]:
scores_rerank = reranker.predict(pairs)

In [212]:
scores_rerank

array([9.74177659e-01, 9.97222781e-01, 1.03043451e-03, 4.43346426e-03,
       1.88001785e-02, 3.17618484e-03, 8.50755349e-03, 3.56394635e-03,
       1.11958226e-02, 2.99951923e-03, 5.61317848e-03, 1.14752650e-02,
       9.02897865e-03, 7.48341763e-03, 1.57273188e-02, 3.38003063e-03,
       3.30215693e-03, 7.75468629e-03, 1.04987770e-02, 7.16966577e-03,
       1.54891126e-02, 1.03537627e-02, 1.20557165e-02, 7.76596135e-03,
       1.12690069e-02, 1.44885248e-02, 1.73418932e-02, 1.55524933e-03,
       5.94203768e-04, 8.93432414e-04, 3.59738804e-03, 6.59633987e-03,
       2.58249440e-03, 8.21081398e-04], dtype=float32)

Añadir scores del reranking a los resultados del retrieval

In [213]:
tripletas_reranked={}

In [214]:
for idx, elem in enumerate(tripletas_formateadas):
    tripletas_reranked[elem] = scores_rerank[idx].item()
    # print(k)

In [215]:
tripletas_reranked

{'The Fascist -> director -> Luciano Salce': 0.9741776585578918,
 'The Fascist -> director -> Luciano Salce -> place_of_birth -> Rome': 0.9972227811813354,
 'The Da Vinci Code (film) -> country_of_origin -> American': 0.0010304345050826669,
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Stephen Roberts (director)': 0.0044334642589092255,
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Diane Gilliam Fisher': 0.018800178542733192,
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Alison Skipper': 0.0031761848367750645,
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Ted Swinford': 0.008507553488016129,
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> McAllister Hull': 0.003563946345821023,
 'The Da Vinci Code (film) -> country_of_origin -> American -> country_of_citizenship -> Oscar Apfel

In [ ]:
def reranking_tripletas(question, tripletas):
    reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)
    pairs = [[question, relacion] for relacion in tripletas ]
    scores_rerank = reranker.predict(pairs)
    tripletas_reranked={}
    for idx, elem in enumerate(tripletas):
        tripletas_reranked[elem] = scores_rerank[idx].item()
    return tripletas_reranked

In [ ]:
def filtrar_tripletas_reranked(tripletas_reranked):
    tripletas_filt = [k for k,v in tripletas_reranked.items() if v > 0.75]
    return tripletas_filt